In [1]:
import sys, os
current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
_='''
read data from semi_processed

select is_good_peak == 3

see interval_id

devid interval to n batch (each k interval) each batch [i1,i2)

write class model that get intreval of interval id [i1,i2) and run model over interval [i1,i2)
model get interval [i3,i4) and test model over it

for each batch train Model
and test it over interval after and before

if error < threshould then merge to batch and train new model and compute error for before and after batch

do this work until no error least than thresould
'''

In [3]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from src.models.utils import *
# Placeholder for your actual model class
class Model:
    def __init__(self, interval, data):
        self.interval = interval
        self.data = data
        self.model = None
        self.n_mimo = 1

    def train(self):
        # TODO: Implement model training on self.data
        
        train_data = get_in(self.data,self.interval)
        train_data = train_data[["generation"]+["temperature"]]#,"humidity", "dew", "surface_pressure"]]
        X = train_data.drop(columns=["generation"])
        y = train_data["generation"]
        
        model = LinearRegression()
        model.fit(X,y)
        self.model = model
        
        #print(f"Training model on interval [{self.i_start}, {self.i_end}) with {len(self.data)} rows")
        

    def test(self, interval):
        y_pred,y = self.pred(interval)
        rmse_error_test = compute_relative_rmse(y_pred, y)
        return rmse_error_test
    
    def pred(self, interval):
        if interval == None : interval = self.interval
        test_data = get_in(self.data,interval)
        test_data = test_data[["generation"]+["temperature"]]#,"humidity", "dew", "surface_pressure"]]
        X = test_data.drop(columns=["generation"])
        y = test_data["generation"]
    
        y_pred = self.model.predict(X)
        return y_pred,y


def split_intervals(intervals, batch_size):
    """Split sorted unique interval IDs into batches of size batch_size"""
    batches = []
    for i in range(0, len(intervals), batch_size):
        batches.append((i,i+batch_size))
    batches[-1] = (batches[-1][0],len(intervals))
    return batches

def get_in(df,i):
    (i1,i2) = i
    mask = (i1 <= df['interval_id']) & (df['interval_id'] < i2)
    return df[mask]

In [4]:
pd.set_option('future.no_silent_downcasting', True)
# Load your data
# TODO: Replace with actual data reading logic
# df = pd.read_csv('semi_processed.csv') or pd.read_parquet('semi_processed.parquet')
csv_semi_processed_path = os.path.join(project_root, "data", "processed", "semi_processed.csv")
df = pd.read_csv(csv_semi_processed_path, encoding='utf-8')

def start(df,name,code):
    # Select rows where is_good_peak == 3
    filtered_df = df[(df["name"] == name)&(df["code"] == code)&(df['is_good_peak'] >= 3)]
    filtered_df = filtered_df[["name","code","datetime","generation",'interval_id',"temperature", "humidity", "dew", "surface_pressure"]]
    # Unique sorted interval IDs
    unique_intervals = sorted(filtered_df['interval_id'].unique())

    batch_size = 10  # For example, number of intervals per batch
    batches = split_intervals(unique_intervals, batch_size)

    batch_models = []
    errors = [[None,None] for i in range(len(batches))]

    for idx, batch_intervals in enumerate(batches):
        
        # Train model on current batch
        batch_model = Model(batch_intervals, filtered_df)
        batch_model.train()

        # Test model on before and after intervals
        error_before = None
        error_after = None

        if idx > 0:
            # Test on previous batch interval
            prev_intervals = batches[idx-1]
            error_before = batch_model.test(prev_intervals)
            errors[idx][0] = error_before

        if idx < len(batches) - 1:
            # Test on next batch interval
            next_intervals = batches[idx+1]
            error_after = batch_model.test(next_intervals)
            errors[idx][1] = error_after

        #print(f"Batch {idx}: error_before={error_before}, error_after={error_after}")

        batch_models.append(batch_model)
        
        clean = lambda x : x if x != None else 100
        errors = [[clean(e[0]),clean(e[1])] for e in errors]
    return batch_models,batches,errors,filtered_df

# Repeat merging & retraining logic until no error less than threshold
# You can wrap above in a loop and add conditions accordingly
# batch_models,batches,errors,filtered_df = start(df)

In [5]:
def merge_batches(batches, batch_models, df, errors, threshold):
    """
    Merge adjacent batches in the 'batches' list when the error between their models is below 'threshold'.
    
    Parameters:
    batches (list of list): Each inner list contains interval_ids representing one batch.
    batch_models (list): List of trained Model instances corresponding to batches.
    df (pd.DataFrame): The dataframe containing the data with 'interval_id' column.
    threshold (float): Error threshold to decide whether to merge batches.
    
    Returns:
    batches (list of list): Updated list of batches after merging.
    batch_models (list): Updated list of models after retraining on merged batches.
    merged (bool): True if any merge was performed, otherwise False.
    """
    
    i = (np.array([error[0] for error in errors])[1:]+np.array([error[1] for error in errors])[:-1]).argmin()
    error_min = errors[i][1]
    if error_min < threshold:
        # Merge batches i and i+1
        if i < len(batches)-1:
            new_batch = (batches[i][0],batches[i+1][1])
        else:
            new_batch = (batches[i][0],batches[i+1][1]) 

        # Train a new model on the merged data
        new_model = Model(new_batch, df)
        new_model.train()
        # Update batches and models lists by replacing merged batches with the new one
        batches[i] = new_batch
        batch_models[i] = new_model
        del batches[i+1]
        del batch_models[i+1]
        
        e2 = batch_models[i].test(batches[i+1]) if i < len(batches)-1 else 100
        e1 = batch_models[i].test(batches[i-1]) if i > 0 else 100
        errors[i] = [e1, e2]
        del errors[i+1]
        
        if i < len(errors)-1:
            errors[i+1][0] = batch_models[i+1].test(batches[i])     
        if i > 0:
            errors[i-1][1] = batch_models[i-1].test(batches[i])
        
        '''
        errors[i] = test new model on pre and next batch
        del errors[i+1]
        errors[i-1][1] = test model[i-1] on new batch
        errors[i+1][0] = test model[i+1] on new batch
        '''
        return True,i,error_min
    return False,None,error_min


In [6]:
import pandas as pd
import plotly.express as px

# Function to assign batch label to each row based on interval_id
def assign_batch(interval_id,batches):
    for i, (start, end) in enumerate(batches):
        if start <= interval_id < end:
            return f'batch_{i+1}'
    return 'out_of_batch'

def show(filtered_df,batches,num=None):
    if num != None : num = [f"batch_{i}" for i in num]
    df_plot = filtered_df.copy()

    assign_batch_m = lambda interval_id : assign_batch(interval_id,batches)
    df_plot['batch'] = df_plot['interval_id'].apply(assign_batch_m)
    #pred_pred(batch_models,df_plot)
    
    if num != None : df_plot = df_plot[df_plot['batch'].isin(num)]
    df_plot
    # Plot with plotly express, color by batch so each batch gets a distinct color
    fig = px.scatter(df_plot, x="datetime", y='generation', color='batch',
                title='Generation over Time by Batch Interval',
                labels={'generation': 'Generation', 'datetime': 'Time'},
                hover_data=['datetime', 'generation', 'temperature'])

    fig.show()
    fig = px.scatter(df_plot, x="temperature", y='generation', color='batch',
                title='Generation over Time by Batch Interval',
                labels={'generation': 'Generation', 'datetime': 'Time'},
                hover_data=['datetime', 'generation', 'temperature'])

    fig.show()
    
def pred_pred(batch_models,df_plot):
    df_plot["prediction"] = None
    for i,model in enumerate(batch_models):
        y1,_ = model.pred(interval=None)
        data = df_plot[df_plot['batch'] == f"batch_{i+1}"]
        print(i,len(y1),len(data),model.interval)
        df_plot.loc[data.index,"prediction"] = y1

In [8]:
name = "پرند"
code = "G11"
batch_models,batches,errors,filtered_df = start(df,name,code)
k = 1
is_merged = True
while len(batch_models) > k and is_merged:
    is_merged,i,error_min = merge_batches(batches, batch_models, filtered_df,errors, threshold=10)
    print(error_min)
    #print(len(errors),len(batches),i)

2.1440745412580635
1.9685358926644771
2.4899184809826833
1.772025491425492
2.142511679075202
2.1544945779950773
2.1354207727131587
2.6212549983812283
2.3551429978873264
2.544805082963195
2.8972887419755007
3.023529764411547
2.57352571389475
2.9700715662131123
2.9540560790538866
2.633574166086551
2.894659613053971
2.885670191421315
3.1272337672284323
3.2681038691095705
3.286995499492238
2.7950289092832623
3.042208470141558
3.0890855604820753
2.9653832834988547
2.552687527480134
2.8114348019141917
2.993775442587463
3.3832826609251723
3.5411982317384925
3.5260796656039326
2.59413844782694
2.2997614772962036
4.598284642075482
5.340763717562281
6.510400401633311
3.051546368897258
5.197477650369056
6.327017510251349
5.43408862321194
3.5902336159107673
9.049992766399448
7.295018409575977
8.608187538152261
3.53318780654523
15.169296978054735


In [9]:
show(filtered_df,batches)

In [ ]:
_='''
is_merged,i,error_min = merge_batches(batches, batch_models, filtered_df,errors, threshold=100)
print(len(errors))
print(error_min)
print(errors)
show(filtered_df,batches)
'''

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


def assign_batch(interval_id,batches):
    for i, (start, end) in enumerate(batches):
        if start <= interval_id < end:
            return i
    return -1

def label(filtered_df,batches):
    df_plot = filtered_df.copy()
    assign_batch_m = lambda interval_id : assign_batch(interval_id,batches)
    df_plot['batch'] = df_plot['interval_id'].apply(assign_batch_m)
    return df_plot.drop(columns=["batch"]),df_plot["batch"]
    
def check(X,y):

    # تعریف مدل SVM با کرنل خطی (Linear Kernel)
    model = SVC(kernel='linear')

    # آموزش مدل
    model.fit(X, y)

    # پیش‌بینی روی داده تست
    y_pred = model.predict(X)
    print("Accuracy:", accuracy_score(y, y_pred))
    return y_pred
    # دقت مدل
    
X,y = label(filtered_df,batches)
X = X[["generation"]+["temperature","humidity", "dew", "surface_pressure"]]
y_pred = check(X,y)
print(classification_report(y, y_pred))
